# Bug hunt: 3-class CT organ classifier (bladder / heart / liver)

**Interviewer:** "This notebook trains a 3-class CT classifier, but it doesn't work and the results look strange. Please walk me through it, tell me what you're checking, and fix it."

There are **10 problems**: 3 crash the code, 7 are silent (one of those crashes only on a GPU).

**How to work**
1. Read the whole notebook first and say out loud what each part does.
2. Run the cells and fix crash errors one at a time.
3. Once it runs, go through it again for the silent bugs.

*Tip: Runtime → Change runtime type → T4 GPU, so you can also catch the GPU-only bug.*

## Setup (no bugs here)
Downloads OrganAMNIST and saves the 3 classes as JPG folders, exactly like `export_jpg.py`:
`/content/data/ct3_jpg/{train,val,test}/{bladder,heart,liver}`

In [ ]:
!pip -q install medmnist

In [ ]:
import os
import numpy as np
from PIL import Image
import medmnist

ROOT = "/content/data"
OUT = os.path.join(ROOT, "ct3_jpg")
KEEP = {0: "bladder", 3: "heart", 6: "liver"}   # original label -> folder name
os.makedirs(ROOT, exist_ok=True)

for split in ["train", "val", "test"]:
    ds = medmnist.OrganAMNIST(split=split, download=True, size=64, root=ROOT)
    imgs = ds.imgs                      # (N, 64, 64) uint8
    labels = ds.labels.reshape(-1)      # (N, 1) -> (N,)
    for old_label, name in KEEP.items():
        folder = os.path.join(OUT, split, name)
        os.makedirs(folder, exist_ok=True)
        idx = np.where(labels == old_label)[0]
        for i in idx:
            Image.fromarray(imgs[i]).save(os.path.join(folder, f"{split}_{i:05d}.jpg"), quality=95)
        print(f"{split:5s} {name:8s} {len(idx):5d} images")

## Bug hunt starts here
### Imports and settings

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

DATA = "/content/data/ct3_jpg"
BATCH_SIZE = 64
EPOCHS = 5
LR = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

### Transforms

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

### Data

In [ ]:
train_ds = datasets.ImageFolder(os.path.join(DATA, "train"), transform=train_tf)
val_ds = datasets.ImageFolder(os.path.join(DATA, "val"), transform=train_tf)
print("classes:", train_ds.classes)
print("train:", len(train_ds), "val:", len(val_ds))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

### Model

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(32 * 16 * 16, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.flatten(1)
        x = self.dropout(x)
        x = self.fc(x)
        return torch.softmax(x, dim=1)


model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

### Training and validation

In [ ]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()
    for images, labels in train_loader:
        images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
    train_loss /= len(train_ds)

    # validation
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item() * images.size(0)
            preds = outputs.argmax(dim=0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_loss /= len(val_loader)
    val_acc = correct / total

    print(f"epoch {epoch}: train loss {train_loss:.3f} | val loss {val_loss:.3f} acc {val_acc:.3f}")

### Your notes
Write each bug like this:
> Bug: cell … → why it's wrong → my fix → crash or silent?